In [ ]:
!pip install kagglehub timm lion-pytorch albumentations tqdm rapidfuzz
import torch
# 1. Check if CUDA is available at all
print(torch.cuda.is_available())

In [ ]:
import os, shutil, random, cv2
import numpy as np
from tqdm import tqdm
from rapidfuzz import fuzz

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [ ]:
import kagglehub

# Dataset 1
#path1 = kagglehub.dataset_download("sujaykapadnis/cucumber-disease-recognition-dataset")
path1=r"E:\internship\disease_dataset\Cucumber Disease Recognition Dataset"
# Dataset 2
#path2 = kagglehub.dataset_download("raiaone/olid-i")
path2=r"E:\internship\olid_dataset\archive"
print("Dataset1:", path1)
print("Dataset2:", path2)


In [ ]:
def find_class_folder(root, prefer="Original"):
    candidate = None

    for root_dir, dirs, files in os.walk(root):
        if len(dirs) > 5 and all(os.path.isdir(os.path.join(root_dir, d)) for d in dirs):

            # If preferred folder name is found, return immediately
            if prefer.lower() in root_dir.lower():
                return root_dir

            # Otherwise store as fallback
            candidate = root_dir

    return candidate


dataset_root = "/root/.cache/kagglehub/datasets/sujaykapadnis/cucumber-disease-recognition-dataset"
SUJAY_PATH = find_class_folder(path1)

print("Detected class folder:", SUJAY_PATH)
print("Classes:", os.listdir(SUJAY_PATH))


In [ ]:
def normalize_name(name):
    name = name.lower()
    name = name.replace("_", " ").replace("-", " ")
    name = "".join(c for c in name if c.isalnum() or c==" ")
    return " ".join(name.split())


In [ ]:
# OLID-I cucumber folder (adjust if nested)
OLID_PATH = path2

# Google Drive dataset
DRIVE_PATH = r"E:\internship\augmented dataset"

# Destination
DEST_ROOT = r"E:\internship\Final_merged"
os.makedirs(DEST_ROOT, exist_ok=True)


In [ ]:
def is_cucumber_class(name):
    name = name.lower()
    return "cucumber" in name
OLID_CLASSES_ALL = [c for c in os.listdir(OLID_PATH) if os.path.isdir(os.path.join(OLID_PATH, c))]

OLID_CUCUMBER_CLASSES = [c for c in OLID_CLASSES_ALL if is_cucumber_class(c)]

print("OLID Cucumber Classes Only:")
for c in OLID_CUCUMBER_CLASSES:
    print(c)


In [ ]:
print(SUJAY_PATH)

In [ ]:
all_classes = set()
def get_classes(path):
    return [c for c in os.listdir(path) if os.path.isdir(os.path.join(path,c))]
ignore_classes = ["belly rot", "fresh cucumber", "pythium fruit rot",'fresh leaf']
# Sujay dataset (already cucumber-only)
for c in get_classes(SUJAY_PATH):
    if c.lower() not in ignore_classes:
        all_classes.add(normalize_name(c))

# Drive dataset (nutrient + diseases দেখা)
for c in get_classes(DRIVE_PATH):
    all_classes.add(normalize_name(c))

# OLID-I ONLY cucumber
for c in OLID_CUCUMBER_CLASSES:
    all_classes.add(normalize_name(c))

print("ALL RAW CLASSES:")
for c in sorted(all_classes):
    print(c)


In [ ]:
import re

def normalize_name(name):
    name = name.lower()
    name = re.sub(r"[_\-]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    # remove noisy words
    name = re.sub(r"(leaf|fruit|cucumber|disease|image|img)", "", name)
    return name.strip()

normalized = {c: normalize_name(c) for c in all_classes}

print("\nNORMALIZED CLASSES:")
for k,v in normalized.items():
    print(k, "→", v)


In [ ]:
!pip install rapidfuzz
from rapidfuzz import fuzz

MERGE_THRESHOLD = 85  # strict for research

final_labels = []
merge_groups = {}

for cls in normalized.values():
    found = False
    for lbl in final_labels:
        if fuzz.ratio(cls, lbl) > MERGE_THRESHOLD:
            merge_groups[lbl].append(cls)
            found = True
            break
    if not found:
        final_labels.append(cls)
        merge_groups[cls] = [cls]

print("\nFINAL MERGED CLASS GROUPS:")
for k,v in merge_groups.items():
    print(k, "<-", set(v))


In [ ]:
# These are the ONLY classes allowed in final dataset
FINAL_ALLOWED_CLASSES = set(merge_groups.keys())
print("FINAL ALLOWED CLASSES:", FINAL_ALLOWED_CLASSES)


In [ ]:
import re

def normalize_name(name):
    name = name.lower()
    name = re.sub(r"[_\-]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"(leaf|fruit|cucumber|disease|image|img)", "", name)
    return name.strip()


In [ ]:
def merge_dataset(source_path, source_name):
    for cls in os.listdir(source_path):
        src_cls = os.path.join(source_path, cls)
        if not os.path.isdir(src_cls):
            continue

        cls_norm = normalize_name(cls)

        # Filter taxonomy
        if cls_norm not in FINAL_ALLOWED_CLASSES:
            print("Skipping", source_name, cls)
            continue

        dest_cls = os.path.join(DEST_ROOT, cls_norm)
        os.makedirs(dest_cls, exist_ok=True)

        for img in os.listdir(src_cls):
            if not img.lower().endswith(('.jpg','.jpeg','.png','.bmp','.tif','.webp')):
                continue
            src_img = os.path.join(src_cls, img)
            dst_img = os.path.join(dest_cls, f"{source_name}_{cls}_{img}")
            shutil.copy2(src_img, dst_img)

merge_dataset(SUJAY_PATH, "sujay")
merge_dataset(DRIVE_PATH, "drive")
merge_dataset(OLID_PATH, "olid")


In [ ]:
!pip install albumentations opencv-python
import albumentations as A
import cv2
import os
import random
import numpy as np

augmentor = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=25, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(p=0.2),
    A.RandomScale(scale_limit=0.2, p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.3)
])


In [ ]:
TARGET_COUNT = 500
DATASET_PATH = DEST_ROOT
IGNORE_CLASSES = ["unclassified"]

def augment_to_target(class_path):
    cls_name = os.path.basename(class_path).lower()

    # Skip ignored classes
    if cls_name in IGNORE_CLASSES:
        print(f"Skipping {cls_name}")
        return

    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg','.png','.jpeg'))]
    count = len(images)

    # Skip empty folders
    if count == 0:
        print(f"⚠️ {cls_name} has 0 images. Skipping.")
        return

    if count >= TARGET_COUNT:
        print(f"{cls_name} already has {count} images")
        return

    print(f"Augmenting {cls_name}: {count} → {TARGET_COUNT}")

    while count < TARGET_COUNT:
        img_name = random.choice(images)
        img_path = os.path.join(class_path, img_name)

        img = cv2.imread(img_path)
        if img is None:
            continue

        augmented = augmentor(image=img)["image"]

        new_name = f"aug_{count}_{img_name}"
        cv2.imwrite(os.path.join(class_path, new_name), augmented)

        count += 1


# Run augmentation
for cls in os.listdir(DATASET_PATH):
    class_dir = os.path.join(DATASET_PATH, cls)
    if os.path.isdir(class_dir):
        augment_to_target(class_dir)


In [ ]:
import os, shutil, random

DATASET_PATH = DEST_ROOT
SPLIT_PATH = r"E:\internship\cucumber_split"
TRAIN_PATH = os.path.join(SPLIT_PATH, "train")
TEST_PATH  = os.path.join(SPLIT_PATH, "test")

os.makedirs(TRAIN_PATH, exist_ok=True)
os.makedirs(TEST_PATH, exist_ok=True)

SPLIT_RATIO = 0.8

for cls in os.listdir(DATASET_PATH):
    cls_dir = os.path.join(DATASET_PATH, cls)
    if not os.path.isdir(cls_dir):
        continue

    images = os.listdir(cls_dir)
    random.shuffle(images)

    split_idx = int(len(images) * SPLIT_RATIO)
    train_imgs = images[:split_idx]
    test_imgs  = images[split_idx:]

    os.makedirs(os.path.join(TRAIN_PATH, cls), exist_ok=True)
    os.makedirs(os.path.join(TEST_PATH, cls), exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(cls_dir, img), os.path.join(TRAIN_PATH, cls, img))
    for img in test_imgs:
        shutil.copy(os.path.join(cls_dir, img), os.path.join(TEST_PATH, cls, img))

print("Train/Test split completed.")


In [ ]:
import os

def remove_empty_class_folders(root):
    for cls in os.listdir(root):
        cls_dir = os.path.join(root, cls)
        if os.path.isdir(cls_dir):
            imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'))]
            if len(imgs) == 0:
                print("Removing empty class:", cls)
                os.rmdir(cls_dir)

remove_empty_class_folders(TRAIN_PATH)
remove_empty_class_folders(TEST_PATH)


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
BATCH_SIZE = 64  # Increased from 32
import multiprocessing

# Get optimal CPU cores for loading (usually 4-8 is best)
workers = min(8, multiprocessing.cpu_count()) 
IMG_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_dataset = datasets.ImageFolder(TRAIN_PATH, transform=transform)
test_dataset  = datasets.ImageFolder(TEST_PATH, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=workers, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=workers, pin_memory=True, persistent_workers=True)

NUM_CLASSES = len(train_dataset.classes)
print("Classes:", train_dataset.classes)
print("Num classes:", NUM_CLASSES)


In [ ]:
!pip install timm lion-pytorch
import timm
import torch.nn as nn
torch.backends.cudnn.benchmark = True 
model = timm.create_model("mobilenetv4_conv_small.e2400_r224_in1k", pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, 14)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)


In [ ]:
from lion_pytorch import Lion
#model = torch.compile(model, mode="reduce-overhead")
 # Optional, can speed up training in PyTorch 2.0+ with compatible models
criterion = nn.CrossEntropyLoss()
optimizer = Lion(model.parameters(), lr=1e-4, weight_decay=1e-2)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
!pip install tqdm
from tqdm import tqdm
import torch
import numpy as np

EPOCHS = 40
PATIENCE = 10

best_val_acc = 0
patience_counter = 0

history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")
    
    # ---------------- TRAIN ---------------- #
    model.train()
    train_loss, correct, total = 0, 0, 0

    train_bar = tqdm(train_loader, desc="Training", leave=False)

    for x, y in train_bar:
        x, y = x.to(device,non_blocking=True), y.to(device,non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            outputs = model(x)
            loss = criterion(outputs, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

        train_bar.set_postfix(loss=loss.item())

    train_loss /= len(train_loader)
    train_acc = correct / total

    # ---------------- VALIDATION ---------------- #
    model.eval()
    val_loss, correct, total = 0, 0, 0

    val_bar = tqdm(test_loader, desc="Validation", leave=False)

    with torch.no_grad():
        for x, y in val_bar:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)

            val_loss += loss.item()
            preds = outputs.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            val_bar.set_postfix(loss=loss.item())

    val_loss /= len(test_loader)
    val_acc = correct / total

    # Save history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    # ---------------- EARLY STOPPING ---------------- #
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), r"E:\internship/mobilenetv4_cucumber_best.pt")
        print("✅ Model Saved")
    else:
        patience_counter += 1
        print(f"Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print("⛔ Early stopping triggered")
        break


In [ ]:
pip install scikit-learn matplotlib seaborn 

In [ ]:
import os, shutil, random, cv2, re, multiprocessing
import numpy as np
from tqdm import tqdm
from rapidfuzz import fuzz

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import kagglehub

# NEW IMPORTS FOR METRICS AND PLOTTING
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# 1. Load the best saved model weights
model.load_state_dict(torch.load("./mobilenetv4_cucumber_best.pt"))
model.eval()
FINAL_ALLOWED_CLASSES = [i for i in FINAL_ALLOWED_CLASSES if i != "unclassified"] # Exclude 'unclassified' if it exists
all_preds = []
all_targets = []

print("\nRunning final evaluation on Test Set...")
with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Evaluating"):
        x = x.to(device, non_blocking=True)
        
        # We don't necessarily need autocast for inference, but it saves memory
        with torch.cuda.amp.autocast():
            outputs = model(x)
            
        preds = outputs.argmax(1).cpu().numpy()
        targets = y.numpy()
        
        all_preds.extend(preds)
        all_targets.extend(targets)

# 2. Calculate Metrics
acc = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds, average='weighted')
rec = recall_score(all_targets, all_preds, average='weighted')
f1 = f1_score(all_targets, all_preds, average='weighted')

print("\n" + "="*40)
print("FINAL CLASSIFICATION METRICS")
print("="*40)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f} (Weighted)")
print(f"Recall:    {rec:.4f} (Weighted)")
print(f"F1 Score:  {f1:.4f} (Weighted)")
print("="*40)

# Print a detailed per-class report
print("\nDetailed Classification Report:")
print(classification_report(all_targets, all_preds, target_names=FINAL_ALLOWED_CLASSES))

# 3. Plot Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=FINAL_ALLOWED_CLASSES, yticklabels=FINAL_ALLOWED_CLASSES, 
            cbar=False, linewidths=0.5)

plt.title('Confusion Matrix: Cucumber Disease Classification', fontsize=16, pad=20)
plt.xlabel('Predicted Label', fontsize=14, labelpad=15)
plt.ylabel('True Label', fontsize=14, labelpad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# Save the plot and show it
plt.savefig('./confusion_matrix.png', dpi=300)
print("\n✅ Confusion matrix saved to './confusion_matrix.png'")
plt.show()